In [24]:
import os
import sys
from pathlib import Path

# Ensure venv packages are available
VENV_SITE = r"D:\Msc_Data_Analytics\Data_Intensive_Scalable_System\CA2\.venv_311\Lib\site-packages"
if VENV_SITE not in sys.path:
    sys.path.insert(0, VENV_SITE)

import sqlite3
import json
import requests
import pandas as pd
from datetime import datetime, timezone
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv

BASE_DIR = Path.cwd().resolve().parent
load_dotenv(BASE_DIR / "config" / ".env")
print("Step 1 (Ingestion): Already executed — data in Azure Blob")
print("Step 2 (Spark):     Already executed — DB at data/hybrid_esports.db")
print("Step 3 (Analysis):  Already executed — charts in reports/")

Step 1 (Ingestion): Already executed — data in Azure Blob
Step 2 (Spark):     Already executed — DB at data/hybrid_esports.db
Step 3 (Analysis):  Already executed — charts in reports/


In [25]:
DB_PATH = BASE_DIR / "data" / "hybrid_esports.db"
EXPORTS = BASE_DIR / "reports" / "exports"
EXPORTS.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(str(DB_PATH))
tables = ["dim_game","dim_cafe","fact_streaming_metrics",
          "fact_review_engagement","fact_hype_metrics","fact_opportunity_scores"]

for t in tables:
    df = pd.read_sql(f"SELECT * FROM {t}", conn)
    df.to_csv(EXPORTS / f"{t}.csv", index=False)
    print(f" {t}.csv exported ({len(df)} rows)")

conn.close()

 dim_game.csv exported (5 rows)
 dim_cafe.csv exported (8 rows)
 fact_streaming_metrics.csv exported (5 rows)
 fact_review_engagement.csv exported (5 rows)
 fact_hype_metrics.csv exported (5 rows)
 fact_opportunity_scores.csv exported (5 rows)


In [26]:
CONN_STR = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
blob_service = BlobServiceClient.from_connection_string(CONN_STR)

# Upload exports to Azure
container = blob_service.get_container_client("processed")
for csv_file in EXPORTS.glob("*.csv"):
    with open(csv_file, "rb") as f:
        container.upload_blob(name=csv_file.name, data=f, overwrite=True)
    print(f" Uploaded to Azure 'processed': {csv_file.name}")

print("\nPipeline COMPLETE. All outputs in Azure Blob + local reports.")

 Uploaded to Azure 'processed': dim_cafe.csv
 Uploaded to Azure 'processed': dim_game.csv
 Uploaded to Azure 'processed': fact_hype_metrics.csv
 Uploaded to Azure 'processed': fact_opportunity_scores.csv
 Uploaded to Azure 'processed': fact_review_engagement.csv
 Uploaded to Azure 'processed': fact_streaming_metrics.csv

Pipeline COMPLETE. All outputs in Azure Blob + local reports.
